# Data Preparation: Train Data
## Version 2

Two datasets are used:

- [Myanmar ALT](https://www2.nict.go.jp/astrec-att/member/mutiyama/ALT/my-alt-190530.zip) from [ALT Treebank Corpus](https://www2.nict.go.jp/astrec-att/member/mutiyama/ALT/)
- [Sayar's myPos version 3.0](https://github.com/ye-kyaw-thu/myPOS/blob/master/corpus-ver-3.0/corpus/mypos-ver.3.0.txt)

These datasets are then preprocessed and normalized into a standardized format for the language model (LM), primarily for KenLM.

## Setup

In [1]:
import pandas as pd

## Load All Datasets

In [2]:
!pwd

/home/lawun330/Desktop/burmese-domain-specific-lm/notebooks


In [3]:
df1 = pd.read_csv(
    "../data/train/my-alt-190530-data",
    sep="\t",
    header=None,
    names=["id", "sentence"]
)
df1.head()

,id,sentence
0,SNT.80188.1,(ROOT (NOUN (NOUN (NOUN (NOUN (noun (noun ပြင်...
1,SNT.80188.2,(ROOT (NOUN (NOUN (noun (noun အန်ဒရီယာ) (noun ...
2,SNT.80188.3,(ROOT (VERB (NOUN (NOUN (noun (adj ပထမ) (noun ...
3,SNT.80188.4,(ROOT (VERB (NOUN (noun ပေါ်တူဂီ) (adp သည်) ) ...
4,SNT.80188.5,(ROOT (VERB (NOUN (NOUN (noun အီတလီ) (adp သည်)...


In [4]:
with open("../data/train/mypos-ver.3.0.shuf.notag.nopunc.txt", encoding="utf-8") as f:
    lines = f.read().splitlines()
df2 = pd.DataFrame({"text": lines})
df2.head()

,text
0,၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရ...
1,လူ တိုင်း တွင် သင့်မြတ် လျော်ကန် စွာ ကန့်သတ် ထ...
2,ဤ နည်း ကို စစ်ယူ သော နည်း ဟု ခေါ် သည်
3,စာပြန်ပွဲ ဆို တာ က အာဂုံဆောင် အလွတ်ကျက် ထား တဲ...
4,ဒီ မှာ ကျွန်တော့် သက်သေခံကတ် ပါ


### Remove Uncessary Data

- The first dataset contains an unnecessary column: `id`.

In [5]:
df1_new = df1.drop(columns=["id"])
df1_new = df1_new.rename(columns={"sentence": "text"})  # optional
df1_new.head()

,text
0,(ROOT (NOUN (NOUN (NOUN (NOUN (noun (noun ပြင်...
1,(ROOT (NOUN (NOUN (noun (noun အန်ဒရီယာ) (noun ...
2,(ROOT (VERB (NOUN (NOUN (noun (adj ပထမ) (noun ...
3,(ROOT (VERB (NOUN (noun ပေါ်တူဂီ) (adp သည်) ) ...
4,(ROOT (VERB (NOUN (NOUN (noun အီတလီ) (adp သည်)...


## Merge All Datasets

In [6]:
print(df1_new.shape)
print(df2.shape)

(20106, 1)
(43196, 1)


In [7]:
df = pd.concat([df1_new, df2], ignore_index=True)
df.head()

,text
0,(ROOT (NOUN (NOUN (NOUN (NOUN (noun (noun ပြင်...
1,(ROOT (NOUN (NOUN (noun (noun အန်ဒရီယာ) (noun ...
2,(ROOT (VERB (NOUN (NOUN (noun (adj ပထမ) (noun ...
3,(ROOT (VERB (NOUN (noun ပေါ်တူဂီ) (adp သည်) ) ...
4,(ROOT (VERB (NOUN (NOUN (noun အီတလီ) (adp သည်)...


In [8]:
print(df.shape)

(63302, 1)


In [9]:
df["text"].to_csv("../data/train/combined_2_raw.txt", index=False, header=False, lineterminator="\n")

## Preprocess All Datasets

### Remove Punctuations and Word Tags

The goal of this project is to use KenLM. Therefore, all annotations are removed for now to create a standard corpus.

In [10]:
!python ../clean_text.py -i ../data/train/combined_2_raw.txt -o ../data/train/combined_2_cleaned.txt

Success! Cleaned text saved to: ../data/train/combined_2_cleaned.txt


## Load Final Dataset

In [11]:
new_df = pd.read_csv("../data/train/combined_2_cleaned.txt", header=None, names=["text"])
new_df

,text
0,ပြင်သစ် နိုင်ငံ ပါရီ မြို့ ပါ့ဒက်စ် ပရင့်စက် ...
1,အန်ဒရီယာ မာစီ သည် အီတလီ အတွက် စမ်းသပ် မှု တစ်...
2,ပထမ တစ်ဝက် ၏ တော်တော်များများ အတွက် ကစား ပွဲ ...
3,ပေါ်တူဂီ သည် ဘယ်သောအခါ မှ စွန့်လွှတ် မှု မ ရှ...
4,အီတလီ သည် ပထမပိုင်း ၌ ၁၆ ၅ ဖြင့် ဦးဆောင် ခဲ့ ...
...,...
63284,အခု ဘာ လုပ် နေ လဲ
63285,ဇူလိုင် ၁၄ ရက် မှာ ဘန်ကောက် ကို သွား မယ့် 123 ...
63286,ကား မှ ကားဘီး ကို ဖြုတ် လိုက် သည်
63287,ကျွန်တော် သိ ပါရစေ


## Exploring the Final Dataset File Size

In [12]:
!ls -lh ../data/train/combined_2_raw.txt ../data/train/combined_2_cleaned.txt

-rw-rw-r-- 1 lawun330 lawun330 18M May 14 19:30 ../data/train/combined_2_cleaned.txt
-rw-rw-r-- 1 lawun330 lawun330 27M May 14 19:30 ../data/train/combined_2_raw.txt
